In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_curve, roc_auc_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [8]:
# Load and clean
df = pd.read_csv("SentimentDataset.csv")
text_cols = ['Text', 'Sentiment', 'Platform', 'Country', 'User', 'Hashtags']
for col in text_cols:
    df[col] = df[col].str.strip()

In [9]:
# Convert multi-class sentiment into binary target
positive_words = ['Positive', 'Joy', 'Excitement', 'Contentment', 'Gratitude',
                   'Happy', 'Hopeful', 'Pride', 'Serenity', 'Awe']
negative_words = ['Negative', 'Sad', 'Grief', 'Despair', 'Loneliness',
                   'Anger', 'Confusion', 'Fear', 'Disgust']

In [10]:
df = df[df['Sentiment'].isin(positive_words + negative_words)].copy()
df['target'] = df['Sentiment'].apply(lambda s: 1 if s in positive_words else 0)
print(f"Rows kept for binary classification: {len(df)}")
print(df['target'].value_counts(), "\n")

Rows kept for binary classification: 276
target
1    217
0     59
Name: count, dtype: int64 



In [11]:
# Convert text into numeric features
vectorizer = CountVectorizer(max_features=300, stop_words='english')
X = vectorizer.fit_transform(df['Text'])
y = df['target']

In [12]:
# Split, train
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [14]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [15]:
# Interpret coefficients / odds ratios
coefs = model.coef_[0]
words = vectorizer.get_feature_names_out()
odds_ratios = np.exp(coefs)
coef_table = pd.DataFrame({'word': words, 'coefficient': coefs, 'odds_ratio': odds_ratios})

print("Top 10 words pushing sentiment POSITIVE:-")
print(coef_table.sort_values('coefficient', ascending=False).head(10).to_string(index=False))
print("\nTop 10 words pushing sentiment NEGATIVE:-")
print(coef_table.sort_values('coefficient').head(10).to_string(index=False))

Top 10 words pushing sentiment POSITIVE:-
       word  coefficient  odds_ratio
        new     1.048509    2.853395
  gratitude     0.736870    2.089386
   serenity     0.733736    2.082848
    friends     0.718915    2.052206
      pride     0.666020    1.946476
 excitement     0.605176    1.831574
  attending     0.593558    1.810419
      music     0.584176    1.793513
contentment     0.542612    1.720495
    weekend     0.508334    1.662519

Top 10 words pushing sentiment NEGATIVE:-
      word  coefficient  odds_ratio
   despair    -2.111262    0.121085
loneliness    -1.754912    0.172922
   feeling    -1.324123    0.266036
 confusion    -1.172269    0.309663
     grief    -0.944544    0.388857
 labyrinth    -0.937186    0.391729
     sense    -0.803619    0.447706
      lost    -0.698598    0.497282
     tears    -0.689943    0.501605
   society    -0.654853    0.519519


In [16]:
# Evaluation
print(f"\nAccuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")

y_proba = model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
print(f"ROC AUC:   {auc:.3f}")


Accuracy:  0.929
Precision: 0.955
Recall:    0.955
ROC AUC:   0.979


In [17]:
plt.figure()
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc:.2f})")
plt.plot([0, 1], [0, 1], linestyle='--', label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Sentiment Classifier")
plt.legend()
plt.savefig("roc_curve.png")
print("\nSaved ROC curve to roc_curve.png")


Saved ROC curve to roc_curve.png
